In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

# Loading in the data

In [2]:
ROOT = Path(".")
IMG_DIR = ROOT / "Image"

FOLDERS = ["Normal", "Stage 1 ROP", "Stage 2 ROP", "Stage 3 ROP", "Stage 4 ROP", "Stage 5 ROP"]

# one row per image file, stacked across class folders
files = pd.concat(
    [pd.DataFrame({"image": [p.name for p in sorted((IMG_DIR / f).iterdir())
                             if p.suffix.lower() in {".jpg", ".jpeg"}],
                   "label": f})
     for f in FOLDERS],
    ignore_index=True,
)
files["path"] = [str(IMG_DIR / lab / name) for lab, name in zip(files["label"], files["image"])]

# attach patient metadata by filename
info = pd.read_excel(ROOT / "img_info.xlsx")
imgs = files.merge(info, on="image", how="left", validate="one_to_one").rename(columns={"image": "file"})
assert imgs["patient_id"].notna().all(), "some images are missing from img_info.xlsx"

SEVERITY = {"Stage 1 ROP": "Mild ROP", "Stage 2 ROP": "Mild ROP", "Stage 3 ROP": "Moderate ROP",
            "Stage 4 ROP": "Severe ROP", "Stage 5 ROP": "Severe ROP"}
imgs["severity"] = imgs["label"].map(SEVERITY)           
imgs["ROP"] = (imgs["label"] != "Normal").astype(int)    

print("imgs:", imgs.shape[0])
print("features:", imgs.columns)

imgs: 1527
features: Index(['file', 'label', 'path', 'patient_id', 'gender', 'gestational_age',
       'birth_weight', 'plurality', 'delivery_mode', 'categories', 'severity',
       'ROP'],
      dtype='str')


# Loading in the images

In [3]:
# Image dimensions when loaded in
size = 512

def load(p, size):
    with Image.open(p) as im:
        return np.asarray(im.convert("RGB").resize(size))

# Opening the jpeg images as numpy arrays with the standardized size
imgs["image"] = imgs["path"].apply(lambda p: load(p, (size,size)))

X_all = np.stack(imgs["image"].values)     
imgs = imgs.drop(columns="image")          

# ROP occurence recognition task
build a model to predict whether an image is normal or ROP

In [4]:
# Parameters
train_size = 0.8

# unique IDs for each class
no_rop_id = imgs[imgs["ROP"] == 0]["patient_id"].unique()
rop_id = imgs[imgs["ROP"] == 1]["patient_id"].unique()
unique_IDs = np.concatenate([no_rop_id, rop_id])

# Stratified random sampling
rng = np.random.default_rng(42)
no_rop_id_t = rng.choice(no_rop_id, size=int(train_size*len(no_rop_id)), replace=False)
rop_id_t = rng.choice(rop_id, size=int(train_size*len(rop_id)), replace=False)

train_IDs = np.concatenate([no_rop_id_t, rop_id_t])
test_IDs = unique_IDs[~pd.Series(unique_IDs).isin(train_IDs)]

# Creating the train and test tests
is_train = imgs["patient_id"].isin(train_IDs)
train, test = imgs[is_train], imgs[~is_train]
X_train, y_train = np.stack(train["image"].values), train["ROP"]
X_test, y_test = np.stack(test["image"].values), test["ROP"]

print("unique IDs:", unique_IDs.shape[0])
print("train IDs:", train_IDs.shape[0])
print("test IDs:", test_IDs.shape[0])
print()
print("train size:", X_train.shape[0])
print("test size:", X_test.shape[0])
print()
print("normal images in training set:", len(y_train[y_train == 0]))
print("rop images in training set:", len(y_train[y_train == 1]))


KeyError: 'image'

# ROP stage recognition task
Build a model to predict the severity of ROP

In [ ]:
# Parameters
train_size = 0.8

# only the ROP images
sev = imgs[imgs["severity"].notna()]

# unique IDs for each class
mild_IDs = sev[sev["severity"] == "Mild ROP"]["patient_id"].unique()
moderate_IDs = sev[sev["severity"] == "Moderate ROP"]["patient_id"].unique()
severe_IDs = sev[sev["severity"] == "Severe ROP"]["patient_id"].unique()
unique_IDs = np.concatenate([mild_IDs, moderate_IDs, severe_IDs])

# Stratified random sampling
rng = np.random.default_rng(42)
mild_IDs_t = rng.choice(mild_IDs, size=int(train_size*len(mild_IDs)), replace=False)
moderate_IDs_t = rng.choice(moderate_IDs, size=int(train_size*len(moderate_IDs)), replace=False)
severe_IDs_t = rng.choice(severe_IDs, size=int(train_size*len(severe_IDs)), replace=False)

train_IDs = np.concatenate([mild_IDs_t, moderate_IDs_t, severe_IDs_t])
test_IDs = unique_IDs[~pd.Series(unique_IDs).isin(train_IDs)]

# Creating the train and test tests
is_train = sev["patient_id"].isin(train_IDs)
train, test = sev[is_train], sev[~is_train]
X_train, y_train = np.stack(train["image"].values), train["severity"]
X_test, y_test = np.stack(test["image"].values), test["severity"]

print("unique IDs:", unique_IDs.shape[0])
print("train IDs:", train_IDs.shape[0])
print("test IDs:", test_IDs.shape[0])
print()
print("train size:", X_train.shape[0])
print("test size:", X_test.shape[0])
print()
print("Mild ROP Images in training set:", len(y_train[y_train == "Mild ROP"]))
print("Moderate ROP Images in training set:", len(y_train[y_train == "Moderate ROP"]))
print("Severe ROP Images in training set:", len(y_train[y_train == "Severe ROP"]))


unique IDs: 626
train IDs: 500
test IDs: 104

train size: 752
test size: 152

Mild ROP Images in training set: 469
Moderate ROP Images in training set: 210
Severe ROP Images in training set: 73
